# Exploratory Data Analysis: E-Commerce Customer Behaviour

**Project type:** EDA mini project (no machine learning)
**Option chosen:** Option 2 - E-Commerce Customer Behaviour

## Objective
In this project I explore an e-commerce dataset to understand how customers shop. I want to find out which product categories bring in the most money, how customers use different devices, whether returning customers behave differently from new ones, and what (if anything) affects customer ratings.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Data and Configuration

In [ ]:
# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 30)

# Load the dataset
df = pd.read_csv("ecommerce_customer_behavior_dataset.csv")
df.head()

## 3. Dataset Overview

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
# Summary statistics for the numeric columns
df.describe().T

In [ ]:
# Summary of the categorical columns
df.describe(include="object").T

**Overview:** The dataset has **5,000 orders and 18 columns**. The columns cover:
- **Customer info:** Age, Gender, City, Is_Returning_Customer
- **Purchase info:** Product_Category, Unit_Price, Quantity, Discount_Amount, Total_Amount, Payment_Method
- **Behaviour on the site:** Device_Type, Session_Duration_Minutes, Pages_Viewed
- **Outcome:** Delivery_Time_Days, Customer_Rating

Each row is one order. The `Date` column is stored as text, so I will convert it in the cleaning step.

## 4. Data Cleaning

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

# Duplicates
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate Order_IDs:", df["Order_ID"].duplicated().sum())

In [ ]:
# Convert Date to datetime and add helper columns
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.to_period("M")

# Check that Total_Amount = Unit_Price * Quantity - Discount_Amount
check = (df["Unit_Price"] * df["Quantity"] - df["Discount_Amount"]) - df["Total_Amount"]
print("Rows where the total does not match:", (check.abs() > 0.01).sum())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

**Cleaning decisions:**
1. **Missing values:** there are none, so nothing had to be filled or dropped.
2. **Duplicates:** there are no duplicate rows and no repeated Order_IDs, so I kept all 5,000 rows.
3. **Data types:** I converted `Date` from text to datetime so I can look at trends over time, and made a `Month` column.
4. **Consistency check:** `Total_Amount` always equals `Unit_Price x Quantity - Discount_Amount`, so the money columns can be trusted.
5. **Ages and ratings** are all in sensible ranges (ages 18 to 75, ratings 1 to 5), so no values were removed.

## 5. Outlier Detection

In [ ]:
# Count outliers with the IQR rule
num_cols = ["Unit_Price", "Total_Amount", "Discount_Amount",
            "Delivery_Time_Days", "Session_Duration_Minutes"]

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: {n} outliers ({n / len(df) * 100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Unit_Price", "Total_Amount", "Delivery_Time_Days"]):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()

In [ ]:
# Where do the extreme prices come from?
df.groupby("Product_Category")["Unit_Price"].agg(["mean", "median", "max"]).round(0).sort_values("mean", ascending=False)

**Outlier decision:** I decided to **keep the outliers**. The extreme `Unit_Price` and `Total_Amount` values come from the **Electronics** category, where items genuinely cost much more than books or food. They are real orders, not mistakes, and removing them would hide the most valuable sales. The high delivery times (up to 25 days) and long sessions are also possible in real life. Because the data is skewed, I use the **median** next to the mean and avoid reading too much into averages alone.

## 6. Exploring Single Variables

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.countplot(data=df, y="Product_Category", order=df["Product_Category"].value_counts().index, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Orders by Product Category")

sns.countplot(data=df, x="Device_Type", order=df["Device_Type"].value_counts().index, ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("Orders by Device Type")

sns.countplot(data=df, y="Payment_Method", order=df["Payment_Method"].value_counts().index, ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title("Orders by Payment Method")

sns.countplot(data=df, y="City", order=df["City"].value_counts().index, ax=axes[1, 1], color="steelblue")
axes[1, 1].set_title("Orders by City")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df["Age"], bins=20, kde=True, ax=axes[0], color="teal")
axes[0].set_title("Age Distribution")

sns.countplot(x="Customer_Rating", data=df, ax=axes[1], color="orange")
axes[1].set_title("Customer Ratings")

sns.histplot(df["Total_Amount"], bins=50, ax=axes[2], color="purple")
axes[2].set_title("Total Amount per Order")

plt.tight_layout()
plt.show()

**Interpretation:**
- Orders are spread almost evenly across the 8 product categories (about 610 to 670 each), so the number of orders does not show which category is most important.
- **Mobile** is the most used device (56% of orders), then Desktop (34%) and Tablet (10%).
- **Credit Card** is the most common payment method (about 40%). Cash on Delivery is the least used (about 5%).
- **Istanbul** has the most orders (about 26%), well ahead of Ankara and Izmir.
- Most customers are in their 20s to 40s (average age is about 35).
- Ratings are mostly positive: about 72% of orders were rated 4 or 5.
- `Total_Amount` is strongly right-skewed: most orders are small and a few are very large.

## 7. Relationships Between Variables

In [ ]:
# Revenue by product category
revenue = df.groupby("Product_Category")["Total_Amount"].sum().sort_values(ascending=False)
print((revenue / revenue.sum() * 100).round(1))

revenue.plot(kind="bar", color="seagreen")
plt.title("Total Revenue by Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Device type and returning customers
print(df.groupby("Device_Type")[["Total_Amount", "Session_Duration_Minutes", "Pages_Viewed"]].mean().round(1))
print()
print(df.groupby("Is_Returning_Customer")[["Total_Amount", "Customer_Rating", "Session_Duration_Minutes"]].mean().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("Device_Type")["Total_Amount"].mean().plot(kind="bar", ax=axes[0], color="coral")
axes[0].set_title("Average Order Value by Device")
df.groupby("Is_Returning_Customer")["Total_Amount"].mean().plot(kind="bar", ax=axes[1], color="slateblue")
axes[1].set_title("Average Order Value: Returning vs New")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly revenue trend
monthly = df.groupby("Month")["Total_Amount"].sum()

monthly.plot(marker="o", figsize=(10, 4), color="crimson")
plt.title("Monthly Revenue")
plt.ylabel("Revenue")
plt.show()

In [ ]:
# What affects customer rating?
print(df.groupby("Delivery_Time_Days")["Customer_Rating"].mean().round(2).head(12))

plt.figure(figsize=(10, 7))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

**Interpretation:**
- **Electronics** makes up almost half of all revenue (about 47%) even though it has about the same number of orders as the other categories. Electronics, Home & Garden and Sports together make about 81% of revenue.
- **Mobile** users have the highest average order value (about 1,006), and Tablet users the lowest (about 921), but the differences between devices are small.
- **Returning and new customers** spend almost the same per order and give almost the same ratings, so being a returning customer does not change behaviour much in this data.
- **Monthly revenue** moves up and down without a clear trend. The strongest months were August 2023, December 2023 and January 2024, and the weakest were April and September 2023. March 2024 is lower mainly because the data stops on 26 March.
- **Customer rating** has almost no correlation with any other column (all values are close to 0). Delivery time, price, discounts and session behaviour do not clearly explain the rating.

## 8. Key Insights

1. **Electronics drives revenue.** It brings in about 47% of total revenue, far more than any other category, because of its high prices.
2. **Order counts are balanced, but revenue is not.** Each category has around 620 orders, so revenue differences come from price, not popularity.
3. **Mobile is the main channel.** 56% of orders come from mobile devices, and mobile users also have the highest average order value.
4. **Returning customers (60% of orders) are not bigger spenders.** They spend and rate about the same as new customers.
5. **Customer satisfaction is high overall** (about 72% gave 4 or 5 stars), but it is not linked to delivery time, price or discount in this dataset.
6. **Istanbul is the biggest market,** with about a quarter of all orders and revenue.
7. **Sales are fairly steady over time,** with small peaks around August, December and January.

## 9. Conclusion

In this project I explored 5,000 e-commerce orders without using any machine learning. The data was clean: no missing values, no duplicates, and the order totals matched the prices, quantities and discounts. I kept the outliers because they are real high-value Electronics orders.

The main finding is that **Electronics is the most important category for revenue**, and **mobile is the most important device**. Returning and new customers behave very similarly, and customer ratings are mostly high but cannot be explained by any single variable in the data.

Based on this, a store should focus on promoting Electronics, make sure the mobile experience is smooth, and look for other factors, such as product quality or customer service, to understand what drives satisfaction.

**Limitations:** this analysis only shows patterns and not causes. The data covers about 15 months and does not include details such as product quality or customer service, which may explain ratings better.